# Lip-sync the host with LatentSync

Re-syncs the **mouth** of the 7 host clips (head motion already looks good) to the reel's voice-over, using
[LatentSync](https://github.com/bytedance/LatentSync) (code Apache-2.0, weights OpenRAIL++ — commercial use allowed).

**Before you start**
1. `Runtime → Change runtime type → T4 GPU` (or any GPU).
2. Upload `lipsync-input.zip` (sent with this notebook) with the file panel on the left, or run the upload cell below.
3. `Runtime → Run all`. Expect roughly **40–90 min on a free T4** (about 5–10 min per ~10 s clip). Finished clips are kept, so if Colab disconnects, reconnect and run the clip cell again — it skips clips that are done.
4. At the end a `lipsync-out.zip` downloads. Send it back.

In [ ]:
# 1. GPU check and model choice
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
mem = int(subprocess.run(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],capture_output=True,text=True).stdout.split()[0])
# LatentSync 1.6 (512 px, sharper) needs ~18 GB VRAM; 1.5 needs ~8 GB. Free Colab T4 (15 GB) -> 1.5.
if mem >= 20000:
    MODEL_REPO, UNET_CFG = 'ByteDance/LatentSync-1.6', 'configs/unet/stage2_512.yaml'
else:
    MODEL_REPO, UNET_CFG = 'ByteDance/LatentSync-1.5', 'configs/unet/stage2.yaml'
print('VRAM MiB:', mem, '->', MODEL_REPO, UNET_CFG)

In [ ]:
# 2. Get the input clips (c0..c6 .mp4 + .wav). Either upload the zip in the file panel to /content, or run this cell to pick it.
import os, glob
if not glob.glob('/content/lipsync-input.zip'):
    from google.colab import files
    up = files.upload()          # choose lipsync-input.zip
!rm -rf /content/input && mkdir -p /content/input && unzip -o -q /content/lipsync-input.zip -d /content/input
# the zip may contain an inner folder; flatten it
!find /content/input -mindepth 2 -type f -exec mv -n {} /content/input/ \;
!ls -l /content/input

In [ ]:
# 3. Install (isolated Python 3.10 venv so LatentSync's pinned versions don't fight Colab's). ~5-8 min.
!apt-get -qq install -y ffmpeg libgl1 > /dev/null
!pip -q install uv
!rm -rf /content/LatentSync && git clone -q https://github.com/bytedance/LatentSync /content/LatentSync
!uv venv --python 3.10 /content/ls-venv
# gradio is only for the web demo, not needed for inference
!sed -i '/^gradio/d' /content/LatentSync/requirements.txt
!cd /content/LatentSync && uv pip install --python /content/ls-venv/bin/python -r requirements.txt --index-strategy unsafe-best-match

In [ ]:
# 4. Download the weights (about 5 GB)
import subprocess
code = f'''
from huggingface_hub import hf_hub_download
for f in ["latentsync_unet.pt", "whisper/tiny.pt"]:
    print(hf_hub_download("{MODEL_REPO}", f, local_dir="/content/LatentSync/checkpoints"))
'''
print(subprocess.run(['/content/ls-venv/bin/python','-c',code],capture_output=True,text=True))

In [ ]:
# 5. Lip-sync every clip. Safe to re-run: finished clips are skipped.
import os, subprocess, time
os.makedirs('/content/out', exist_ok=True)
STEPS, GUIDANCE = 20, 2.0     # more steps = nicer but slower; higher guidance = stronger lip sync (1.0-3.0, >2.5 can jitter)
for i in range(7):
    vid, aud, out = f'/content/input/c{i}.mp4', f'/content/input/c{i}.wav', f'/content/out/l{i}.mp4'
    if os.path.exists(out) and os.path.getsize(out) > 10000:
        print(f'c{i}: already done'); continue
    t = time.time()
    r = subprocess.run(['/content/ls-venv/bin/python','-m','scripts.inference',
        '--unet_config_path', UNET_CFG, '--inference_ckpt_path','checkpoints/latentsync_unet.pt',
        '--inference_steps',str(STEPS),'--guidance_scale',str(GUIDANCE),'--enable_deepcache',
        '--video_path',vid,'--audio_path',aud,'--video_out_path',out,'--temp_dir',f'/content/tmp{i}'],
        cwd='/content/LatentSync', capture_output=True, text=True)
    print(f'c{i}: exit {r.returncode} in {time.time()-t:.0f}s')
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-3000:]); break

In [ ]:
# 6. Quick look at the first result (mouth should move clearly with the speech)
from IPython.display import Video
Video('/content/out/l0.mp4', embed=True, width=360)

In [ ]:
# 7. Zip and download
!cd /content && rm -f lipsync-out.zip && zip -q lipsync-out.zip out/l*.mp4 && ls -l lipsync-out.zip
from google.colab import files
files.download('/content/lipsync-out.zip')

**If something fails**
- *Out of memory*: the clip cell uses 1.5 on a 15 GB GPU. Restart the runtime and re-run; do not force 1.6 on a T4.
- *Install error in step 3*: re-run that cell once (network hiccups), then step 4.
- *Face not detected*: tell me which clip; the head is always frontal in these clips so it should not happen.
- Colab sessions end after a few hours: results in `/content/out` are lost then, so download as soon as step 7 finishes. To be safe you can copy `/content/out` to Google Drive after each clip.